# Target samples

Lists of known astronomical objects for inspecting the OVRO-LWA metacatalog in
[`metacatalog_query.ipynb`](metacatalog_query.ipynb). Paste a row's `coord` string
(decimal degrees, `RA DEC`) into that notebook's nearest-source box.

This notebook does **not** load the metacatalog. Each section downloads (or reuses a
cached copy of) a published catalog and shows it in a Panel **Tabulator** (sortable
columns, per-column header filters). Paste a row's `coord` string into
`metacatalog_query.ipynb`.

| Section | Sample | Source |
| ------- | ------ | ------ |
| Local Galaxies | NED-LVS galaxies with Mstar or SFR, z ≤ 0.05, nearest first | [NED-LVS](https://ned.ipac.caltech.edu/NED::LVS/) (Cook et al. 2023); CLU lineage |
| HRS–GLEAM galaxies | Takeuchi et al. (2026) Table 1 (18 HRS galaxies with GLEAM SEDs) | [arXiv:2204.00831](https://arxiv.org/html/2204.00831v2) |
| Galaxy Clusters | MCXC X-ray clusters, nearest first | [Piffaretti et al. 2011](https://cdsarc.cds.unistra.fr/viz-bin/cat/J/A+A/534/A109) |
| Giant radio sources | Kuźmicz et al. (2018) GRS catalogue (D ≥ 0.7 Mpc) | [VizieR J/ApJS/238/9](https://cdsarc.cds.unistra.fr/viz-bin/cat/J/ApJS/238/9) |
| Peaked-spectrum | Callingham et al. (2017) GLEAM peaked-spectrum sample | [VizieR J/ApJ/836/174](https://cdsarc.cds.unistra.fr/viz-bin/cat/J/ApJ/836/174) |
| Supernova Remnants | Green Galactic SNR catalogue | [Green 2024 Oct / 2025](https://www.mrao.cam.ac.uk/surveys/snrs/) |
| Pulsars | ATNF Pulsar Catalogue | [Manchester et al. 2005](https://www.atnf.csiro.au/research/pulsar/psrcat/) |
| Pulsar Wind Nebulae | Roberts PWNCat (with / without detected pulsar) | [Roberts 2004 / March 2005](https://www.physics.mcgill.ca/~pulsar/pwncat.html) |
| X-ray binaries | Tübingen LMXB + HMXB catalogues | [LMXBcat](http://astro.uni-tuebingen.de/~xrbcat/LMXBcat/january_24/LMXBcat.csv); [HMXBcat](http://astro.uni-tuebingen.de/~xrbcat/HMXBcat/March2025/HMXBcat.csv) |
| X-ray/optical systems | Rodriguez (2024) Table 2 | [arXiv:2401.09537](https://arxiv.org/html/2401.09537v1#A2) |

**Run cells in order.**


In [1]:
from __future__ import annotations

import re
import tarfile
import urllib.request
import gzip
from pathlib import Path

import astropy.units as u
import numpy as np
import pandas as pd
import panel as pn
from astropy.coordinates import BarycentricMeanEcliptic, SkyCoord
from astropy.io import ascii

from lwa_catalog.analyze import load_nedlvs_catalog
from lwa_catalog.constants import NEDLVS_DEFAULT_PATH, REFERENCE_CATALOGS_DIR

pn.extension("tabulator")

# --- operator config -------------------------------------------------------
CACHE_DIR = Path(REFERENCE_CATALOGS_DIR)
NEDLVS_PATH = NEDLVS_DEFAULT_PATH
MAX_REDSHIFT_GALAXIES = 0.05  # ~200 Mpc
MIN_DIST_MPC = 0.05  # 50 kpc; drops Galactic-star contaminants with z ~ 0
MAX_REDSHIFT_GRS = None  # e.g. 0.1 for a local Kuźmicz+2018 subset; None = full catalogue
TABLE_HEIGHT = 420
TABLE_PAGE_SIZE = 25
TABLE_REMOTE_THRESHOLD = 5000  # remote pagination above this many rows
USER_AGENT = "lwa-catalog/target_samples (claw@astro.caltech.edu)"

CACHE_DIR.mkdir(parents=True, exist_ok=True)


def cache_url(url: str, dest: Path, *, force: bool = False) -> Path:
    '''Download *url* to *dest* unless the file already exists.'''
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.is_file() and dest.stat().st_size > 0 and not force:
        print(f"cached {dest} ({dest.stat().st_size:,} bytes)")
        return dest
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=120) as resp:
        dest.write_bytes(resp.read())
    print(f"wrote {dest} ({dest.stat().st_size:,} bytes)")
    return dest


def read_cds(data_path: Path, readme_path: Path):
    '''Read a CDS/VizieR fixed-width table using its ReadMe.'''
    return ascii.read(str(data_path), readme=str(readme_path))


def add_coord(df: pd.DataFrame) -> pd.DataFrame:
    '''Add a `coord` column (`RA DEC` decimal degrees) for metacatalog_query.'''
    out = df.copy()
    out["coord"] = [
        f"{float(ra):.6f} {float(dec):.6f}" for ra, dec in zip(out["RA"], out["DEC"], strict=True)
    ]
    return out


def show_table(df: pd.DataFrame, *, height: int | None = None) -> pn.widgets.Tabulator:
    '''Panel Tabulator with sortable columns and per-column header filters.'''
    print(f"{len(df)} rows, columns: {list(df.columns)}")
    n = len(df)
    if n <= 12:
        pagination = None
        page_size = n
        height = height or min(TABLE_HEIGHT, 80 + 28 * max(n, 1))
    elif n > TABLE_REMOTE_THRESHOLD:
        pagination = "remote"
        page_size = TABLE_PAGE_SIZE
        height = height or TABLE_HEIGHT
    else:
        pagination = "local"
        page_size = TABLE_PAGE_SIZE
        height = height or TABLE_HEIGHT
    return pn.widgets.Tabulator(
        df,
        pagination=pagination,
        page_size=page_size,
        height=height,
        sizing_mode="stretch_width",
        layout="fit_data_table",
        header_filters=True,
        show_index=False,
        disabled=True,
        sortable=True,
        selectable=1,
    )


print("CACHE_DIR =", CACHE_DIR.resolve())
print("NEDLVS_PATH =", Path(NEDLVS_PATH).resolve(), "(exists)" if Path(NEDLVS_PATH).is_file() else "(missing)")
print("MAX_REDSHIFT_GALAXIES =", MAX_REDSHIFT_GALAXIES)
print("MIN_DIST_MPC =", MIN_DIST_MPC)
print("MAX_REDSHIFT_GRS =", MAX_REDSHIFT_GRS)


CACHE_DIR = /fast/claw/catalogs
NEDLVS_PATH = /fast/claw/catalogs/NEDLVS_current.fits (exists)
MAX_REDSHIFT_GALAXIES = 0.05
MIN_DIST_MPC = 0.05
MAX_REDSHIFT_GRS = None


## Local Galaxies

The [Census of the Local Universe](https://ui.adsabs.harvard.edu/abs/2019ApJ...880....7C)
(CLU; Cook et al. 2019) is a nearby-galaxy compilation for gravitational-wave
follow-up. This project already holds the all-sky successor sample, the
[NED Local Volume Sample](https://ned.ipac.caltech.edu/NED::LVS/) (NED-LVS;
Cook et al. 2023), at `NEDLVS_PATH`. NED-LVS includes the CLU galaxies and
extends to D ~ 1000 Mpc.

Keep `objtype == "G"` with catalog redshift 0 ≤ z ≤ `MAX_REDSHIFT_GALAXIES`
(default 0.05, ~200 Mpc), `DistMpc >= MIN_DIST_MPC` (default 0.05 Mpc = 50 kpc,
so the Magellanic Clouds remain and Galactic stars with z ~ 0 are dropped), and a
measured `Mstar` **or** SFR (`SFR_hybrid` or `SFR_W4`). Sort by NED-LVS `DistMpc`
(redshift-independent distances preferred below 200 Mpc).

The full sorted table is `local_galaxies`. Paste `coord` into `metacatalog_query.ipynb`.


In [2]:
ned = load_nedlvs_catalog(NEDLVS_PATH)
z = pd.to_numeric(ned["z"], errors="coerce")
dist = pd.to_numeric(ned["DistMpc"], errors="coerce")
mstar = pd.to_numeric(ned["Mstar"], errors="coerce")
sfr_hybrid = pd.to_numeric(ned["SFR_hybrid"], errors="coerce")
sfr_w4 = pd.to_numeric(ned["SFR_W4"], errors="coerce")
is_galaxy = ned["objtype"].astype(str).str.strip().eq("G")
has_mstar_or_sfr = (
    np.isfinite(mstar.to_numpy())
    | np.isfinite(sfr_hybrid.to_numpy())
    | np.isfinite(sfr_w4.to_numpy())
)
names = ned["objname"].map(
    lambda x: x.decode("utf-8", errors="replace") if isinstance(x, (bytes, bytearray)) else str(x)
)
local_galaxies = (
    ned.loc[
        is_galaxy
        & np.isfinite(z.to_numpy())
        & (z >= 0.0)
        & (z <= MAX_REDSHIFT_GALAXIES)
        & has_mstar_or_sfr
    ]
    .assign(
        name=names,
        DistMpc=dist,
        z=z,
        Mstar=mstar,
        SFR_hybrid=sfr_hybrid,
        SFR_W4=sfr_w4,
    )
)
local_galaxies = local_galaxies.loc[local_galaxies["DistMpc"] >= MIN_DIST_MPC].copy()
local_galaxies = local_galaxies.sort_values(
    "DistMpc", na_position="last", kind="mergesort"
).reset_index(drop=True)
local_galaxies = add_coord(local_galaxies)
local_galaxies = local_galaxies[
    ["name", "RA", "DEC", "coord", "z", "DistMpc", "Diam_arcsec", "Mstar", "SFR_hybrid", "SFR_W4"]
]

print(
    f"NED-LVS galaxies with z <= {MAX_REDSHIFT_GALAXIES} and Mstar/SFR: {len(local_galaxies):,}"
)
print(
    "DistMpc: min={:.3f}  median={:.1f}  max={:.1f}".format(
        local_galaxies["DistMpc"].min(),
        local_galaxies["DistMpc"].median(),
        local_galaxies["DistMpc"].max(),
    )
)
print(
    "finite Mstar={:,}  SFR_hybrid={:,}  SFR_W4={:,}".format(
        int(np.isfinite(local_galaxies["Mstar"]).sum()),
        int(np.isfinite(local_galaxies["SFR_hybrid"]).sum()),
        int(np.isfinite(local_galaxies["SFR_W4"]).sum()),
    )
)
show_table(local_galaxies)


NED-LVS galaxies with z <= 0.05 and Mstar/SFR: 252,175
DistMpc: min=0.050  median=153.3  max=944.5
finite Mstar=252,173  SFR_hybrid=130,508  SFR_W4=251,781
252175 rows, columns: ['name', 'RA', 'DEC', 'coord', 'z', 'DistMpc', 'Diam_arcsec', 'Mstar', 'SFR_hybrid', 'SFR_W4']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='remote', show_index=False, sizing_mode='stretch_width', value=              ...)

## HRS–GLEAM galaxies

[Takeuchi et al. 2026, arXiv:2204.00831](https://arxiv.org/html/2204.00831v2)
(*Estimating Galaxy Star Formation Rates from Murchison Widefield Array Radio
Continuum Emission*) select **18** nearby Herschel Reference Survey (HRS)
galaxies with GLEAM detections for a low-frequency IR–radio correlation /
radio-SFR calibration. Table 1 lists HRS/NGC IDs, ICRS positions, morphology,
optical size \(D_{25}\), distance, GLEAM ID, and optical AGN class.

Three objects (HRS 144, 163, 220) are Seyfert/LINER and are excluded from their
clean star-forming calibration subsample of 15; all 18 are retained here for
sky inspection. The table is `hrs_gleam_galaxies`, sorted by distance.
Paste `coord` into `metacatalog_query.ipynb`.

In [3]:
# Takeuchi et al. 2026 (arXiv:2204.00831v2) Table 1 — HRS galaxies with GLEAM SEDs.
# Positions converted from the paper's sexagesimal R.A./Dec. to ICRS degrees.
# AGN: optical BPT class (empty = star-forming / unclassified in their Table 1).
HRS_GLEAM_ROWS = [
    ("HRS 25", 3437, 163.148958, 22.934139, "Sc", 2.51, 18.24, "J105236+225606", ""),
    ("HRS 36", 3504, 165.796708, 27.972500, "Sab", 2.69, 21.94, "J110311+275812", ""),
    ("HRS 50", 3655, 170.727583, 16.590139, "Sc", 1.55, 21.43, "J112254+163522", ""),
    ("HRS 77", 4030, 180.098500, -1.100000, "Sbc", 4.17, 20.83, "J120023-010607", ""),
    ("HRS 102", 4254, 184.706792, 14.416500, "Sc", 6.15, 17.00, "J121850+142515", ""),
    ("HRS 114", 4303, 185.478750, 4.473639, "Sbc", 6.59, 17.00, "J122154+042827", ""),
    ("HRS 122", 4321, 185.728750, 15.822389, "Sbc", 9.12, 17.00, "J122255+154939", ""),
    ("HRS 144", 4388, 186.445083, 12.662083, "Sb", 5.10, 17.00, "J122548+123917", "Seyfert"),
    ("HRS 163", 4438, 186.939958, 13.008833, "Sb", 8.12, 17.00, "J122744+130020", "Seyfert"),
    ("HRS 190", 4501, 187.996750, 14.420417, "Sb", 7.23, 17.00, "J123159+142503", ""),
    ("HRS 201", 4527, 188.535417, 2.653806, "Sbc", 5.86, 17.00, "J123408+023909", ""),
    ("HRS 203", 4532, 188.580542, 6.467694, "Im", 2.60, 17.00, "J123420+062758", ""),
    ("HRS 204", 4535, 188.584625, 8.197750, "Sc", 8.33, 17.00, "J123418+081157", ""),
    ("HRS 205", 4536, 188.613042, 2.187889, "Sbc", 7.23, 17.00, "J123427+021114", ""),
    ("HRS 220", 4579, 189.431333, 11.818194, "Sb", 6.29, 17.00, "J123743+114909", "LINER"),
    ("HRS 247", 4654, 190.985750, 13.126667, "Scd", 4.99, 17.00, "J124355+130801", ""),
    ("HRS 251", 4666, 191.285792, -0.461889, "Sc", 4.57, 21.61, "J124508-002747", ""),
    ("HRS 306", 5363, 209.030042, 5.254778, "pec", 4.07, 16.23, "J135607+051516", ""),
]
hrs_gleam_galaxies = pd.DataFrame(
    HRS_GLEAM_ROWS,
    columns=["hrs_id", "NGC", "RA", "DEC", "type", "D25_arcmin", "Dist_Mpc", "gleam_id", "AGN"],
)
hrs_gleam_galaxies["name"] = hrs_gleam_galaxies["NGC"].map(lambda n: f"NGC {n}")
hrs_gleam_galaxies = hrs_gleam_galaxies.sort_values(
    "Dist_Mpc", kind="mergesort"
).reset_index(drop=True)
hrs_gleam_galaxies = add_coord(hrs_gleam_galaxies)
hrs_gleam_galaxies = hrs_gleam_galaxies[
    ["name", "hrs_id", "NGC", "RA", "DEC", "coord", "type", "D25_arcmin", "Dist_Mpc", "gleam_id", "AGN"]
]
n_agn = int((hrs_gleam_galaxies["AGN"].astype(str).str.len() > 0).sum())
print(
    f"{len(hrs_gleam_galaxies)} HRS–GLEAM galaxies "
    f"({len(hrs_gleam_galaxies) - n_agn} star-forming, {n_agn} Seyfert/LINER)"
)
print(
    "Dist_Mpc: min={:.2f}  median={:.2f}  max={:.2f}".format(
        hrs_gleam_galaxies["Dist_Mpc"].min(),
        hrs_gleam_galaxies["Dist_Mpc"].median(),
        hrs_gleam_galaxies["Dist_Mpc"].max(),
    )
)
show_table(hrs_gleam_galaxies)

18 HRS–GLEAM galaxies (15 star-forming, 3 Seyfert/LINER)
Dist_Mpc: min=16.23  median=17.00  max=21.94
18 rows, columns: ['name', 'hrs_id', 'NGC', 'RA', 'DEC', 'coord', 'type', 'D25_arcmin', 'Dist_Mpc', 'gleam_id', 'AGN']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=        name  ...)

## Galaxy Clusters

X-ray clusters from the [MCXC meta-catalogue](https://cdsarc.cds.unistra.fr/viz-bin/cat/J/A+A/534/A109)
(Piffaretti et al. 2011; VizieR `J/A+A/534/A109`). MCXC homogenises ROSAT All-Sky
Survey and serendipitous cluster catalogues to Δ=500 masses and radii.

Sorted by redshift (nearest first). The full table is `galaxy_clusters`.


In [4]:
mcxc_dir = CACHE_DIR / "mcxc"
cache_url("https://cdsarc.cds.unistra.fr/ftp/cats/J/A+A/534/A109/ReadMe", mcxc_dir / "ReadMe")
cache_url("https://cdsarc.cds.unistra.fr/ftp/cats/J/A+A/534/A109/mcxc.dat", mcxc_dir / "mcxc.dat")
mcxc = read_cds(mcxc_dir / "mcxc.dat", mcxc_dir / "ReadMe").to_pandas()

aname = mcxc["AName"].astype(str).str.strip().replace({"": pd.NA, "nan": pd.NA})
oname = mcxc["OName"].astype(str).str.strip().replace({"": pd.NA, "nan": pd.NA})
galaxy_clusters = pd.DataFrame(
    {
        "name": aname.fillna(oname).fillna(mcxc["MCXC"].astype(str)),
        "MCXC": mcxc["MCXC"].astype(str),
        "OName": mcxc["OName"].astype(str).str.strip(),
        "RA": pd.to_numeric(mcxc["RAdeg"], errors="coerce"),
        "DEC": pd.to_numeric(mcxc["DEdeg"], errors="coerce"),
        "z": pd.to_numeric(mcxc["z"], errors="coerce"),
        "M500": pd.to_numeric(mcxc["M500"], errors="coerce"),
        "R500_Mpc": pd.to_numeric(mcxc["R500"], errors="coerce"),
        "L500": pd.to_numeric(mcxc["L500"], errors="coerce"),
        "Cat": mcxc["Cat"].astype(str).str.strip(),
    }
)
galaxy_clusters = (
    galaxy_clusters.dropna(subset=["RA", "DEC"])
    .sort_values("z", na_position="last", kind="mergesort")
    .reset_index(drop=True)
)
galaxy_clusters = add_coord(galaxy_clusters)
galaxy_clusters = galaxy_clusters[
    ["name", "MCXC", "OName", "RA", "DEC", "coord", "z", "M500", "R500_Mpc", "L500", "Cat"]
]
show_table(galaxy_clusters)


cached /fast/claw/catalogs/mcxc/ReadMe (7,064 bytes)
cached /fast/claw/catalogs/mcxc/mcxc.dat (415,572 bytes)
1743 rows, columns: ['name', 'MCXC', 'OName', 'RA', 'DEC', 'coord', 'z', 'M500', 'R500_Mpc', 'L500', 'Cat']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=                          ...)

## Giant radio sources

[Kuźmicz et al. 2018](https://ui.adsabs.harvard.edu/abs/2018ApJS..238....9K)
(*An Updated Catalog of Giant Radio Sources*; ApJS 238, 9) compile **349**
literature giant radio sources (galaxies and quasars) with projected linear
size \(D \ge 0.7\) Mpc (up to \(\sim4.7\) Mpc), \(0.016 < z < 3.22\).
Machine-readable table: VizieR
[`J/ApJS/238/9`](https://cdsarc.cds.unistra.fr/viz-bin/cat/J/ApJS/238/9).

Host positions are ICRS. Set `MAX_REDSHIFT_GRS` in the config cell for a local
subset (e.g. `0.1`); `None` keeps the full catalogue. Sorted by projected size
`D_Mpc` (largest first). The table is `giant_radio_sources`.


In [5]:
grs_dir = CACHE_DIR / "kuzmicz2018"
cache_url(
    "https://cdsarc.cds.unistra.fr/ftp/cats/J/ApJS/238/9/ReadMe",
    grs_dir / "ReadMe",
)
cache_url(
    "https://cdsarc.cds.unistra.fr/ftp/cats/J/ApJS/238/9/table1.dat",
    grs_dir / "table1.dat",
)
grs = read_cds(grs_dir / "table1.dat", grs_dir / "ReadMe")
sign = np.where(np.asarray(grs["DE-"], dtype=str) == "-", -1.0, 1.0)
ra = (
    np.asarray(grs["RAh"], dtype=float)
    + np.asarray(grs["RAm"], dtype=float) / 60.0
    + np.asarray(grs["RAs"], dtype=float) / 3600.0
) * 15.0
dec = sign * (
    np.asarray(grs["DEd"], dtype=float)
    + np.asarray(grs["DEm"], dtype=float) / 60.0
    + np.asarray(grs["DEs"], dtype=float) / 3600.0
)
z = pd.to_numeric(np.asarray(grs["z"]), errors="coerce")
giant_radio_sources = pd.DataFrame(
    {
        "name": np.asarray(grs["ID"], dtype=str),
        "OID": pd.Series(np.asarray(grs["OID"], dtype=str)).str.strip(),
        "RA": ra,
        "DEC": dec,
        "z": z,
        "z_phot": pd.Series(np.asarray(grs["n_z"], dtype=str)).str.strip().eq("p"),
        "FR": pd.Series(np.asarray(grs["FR"], dtype=str)).str.strip(),
        "LAS_arcmin": pd.to_numeric(np.asarray(grs["LAS"]), errors="coerce"),
        "D_Mpc": pd.to_numeric(np.asarray(grs["D"]), errors="coerce"),
        "rmag": pd.to_numeric(np.asarray(grs["rmag"]), errors="coerce"),
        "S14_mJy": pd.to_numeric(np.asarray(grs["SI"]), errors="coerce"),
        "logP_WHz": pd.to_numeric(np.asarray(grs["logP"]), errors="coerce"),
    }
)
if MAX_REDSHIFT_GRS is not None:
    giant_radio_sources = giant_radio_sources.loc[
        np.isfinite(giant_radio_sources["z"]) & (giant_radio_sources["z"] <= MAX_REDSHIFT_GRS)
    ].copy()
giant_radio_sources = (
    giant_radio_sources.sort_values("D_Mpc", ascending=False, na_position="last", kind="mergesort")
    .reset_index(drop=True)
)
giant_radio_sources = add_coord(giant_radio_sources)
giant_radio_sources = giant_radio_sources[
    [
        "name",
        "OID",
        "RA",
        "DEC",
        "coord",
        "z",
        "z_phot",
        "FR",
        "LAS_arcmin",
        "D_Mpc",
        "rmag",
        "S14_mJy",
        "logP_WHz",
    ]
]
zmax = "all z" if MAX_REDSHIFT_GRS is None else f"z <= {MAX_REDSHIFT_GRS}"
print(f"Kuźmicz+2018 giant radio sources ({zmax}): {len(giant_radio_sources)}")
print(
    "D_Mpc: min={:.2f}  median={:.2f}  max={:.2f}".format(
        giant_radio_sources["D_Mpc"].min(),
        giant_radio_sources["D_Mpc"].median(),
        giant_radio_sources["D_Mpc"].max(),
    )
)
show_table(giant_radio_sources)


cached /fast/claw/catalogs/kuzmicz2018/ReadMe (9,457 bytes)
cached /fast/claw/catalogs/kuzmicz2018/table1.dat (35,897 bytes)
Kuźmicz+2018 giant radio sources (all z): 349
D_Mpc: min=0.70  median=1.14  max=4.69
349 rows, columns: ['name', 'OID', 'RA', 'DEC', 'coord', 'z', 'z_phot', 'FR', 'LAS_arcmin', 'D_Mpc', 'rmag', 'S14_mJy', 'logP_WHz']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=           name O...)

## Peaked-spectrum sources

[Callingham et al. 2017](https://ui.adsabs.harvard.edu/abs/2017ApJ...836..174C)
(*Extragalactic Peaked-spectrum Radio Sources at Low Frequencies*; ApJ 836, 174)
select peaked-spectrum candidates from GLEAM (72–231 MHz) plus NVSS/SUMSS.
Machine-readable tables: VizieR
[`J/ApJ/836/174`](https://cdsarc.cds.unistra.fr/viz-bin/cat/J/ApJ/836/174).

Combined table is `peaked_spectrum` with `sample`:

| `sample` | VizieR file | Meaning |
| -------- | ----------- | ------- |
| `pkfreq` | `pkfreq.dat` | Spectral peak between 72 MHz and 1.4 GHz (1222) |
| `gps` | `gps.dat` | Peak above 843 MHz (GPS-like; 261) |
| `convex` | `convex.dat` | Convex spectrum in GLEAM→SUMSS/NVSS (116) |
| `pk72` | `pk72.dat` | Peak below 72 MHz (36) |

`pkfreq`+`gps` are the paper's 1483 peaked-spectrum candidates. Sorted by peak
flux `Spk_Jy` (then `S200_Jy`). Paste `coord` into `metacatalog_query.ipynb`.


In [6]:
def _cache_gunzip(url: str, gz_dest: Path, dat_dest: Path, *, force: bool = False) -> Path:
    '''Download a .gz file and write the decompressed companion if needed.'''
    if dat_dest.is_file() and dat_dest.stat().st_size > 0 and not force:
        print(f"cached {dat_dest} ({dat_dest.stat().st_size:,} bytes)")
        return dat_dest
    cache_url(url, gz_dest, force=force)
    dat_dest.write_bytes(gzip.decompress(gz_dest.read_bytes()))
    print(f"wrote {dat_dest} ({dat_dest.stat().st_size:,} bytes)")
    return dat_dest


def load_callingham_table(path: Path, *, sample: str, readme: Path) -> pd.DataFrame:
    '''Load one Callingham+2017 VizieR table into a common schema.'''
    raw = read_cds(path, readme).to_pandas()
    has_peak = "Spk" in raw.columns and "nuPk" in raw.columns
    out = pd.DataFrame(
        {
            "name": raw["GLEAM"].astype(str).str.strip(),
            "sample": sample,
            "RA": pd.to_numeric(raw["RAdeg"], errors="coerce"),
            "DEC": pd.to_numeric(raw["DEdeg"], errors="coerce"),
            "Spk_Jy": pd.to_numeric(raw["Spk"], errors="coerce") if has_peak else pd.Series(np.nan, index=raw.index),
            "nuPk_MHz": pd.to_numeric(raw["nuPk"], errors="coerce") if has_peak else pd.Series(np.nan, index=raw.index),
            "S200_Jy": pd.to_numeric(raw["S200"], errors="coerce"),
            "alpLow": pd.to_numeric(raw["alpLow"], errors="coerce"),
            "alpHigh": pd.to_numeric(raw["alpHigh"], errors="coerce"),
            "alpTn": pd.to_numeric(raw["alpTn"], errors="coerce") if "alpTn" in raw.columns else pd.Series(np.nan, index=raw.index),
            "alpTk": pd.to_numeric(raw["alpTk"], errors="coerce") if "alpTk" in raw.columns else pd.Series(np.nan, index=raw.index),
            "q": pd.to_numeric(raw["q"], errors="coerce"),
            "z": pd.to_numeric(raw["z"], errors="coerce"),
        }
    )
    return out.dropna(subset=["RA", "DEC"]).reset_index(drop=True)


callingham_dir = CACHE_DIR / "callingham2017"
callingham_readme = cache_url(
    "https://cdsarc.cds.unistra.fr/ftp/cats/J/ApJ/836/174/ReadMe",
    callingham_dir / "ReadMe",
)
_cache_gunzip(
    "https://cdsarc.cds.unistra.fr/ftp/cats/J/ApJ/836/174/pkfreq.dat.gz",
    callingham_dir / "pkfreq.dat.gz",
    callingham_dir / "pkfreq.dat",
)
cache_url(
    "https://cdsarc.cds.unistra.fr/ftp/cats/J/ApJ/836/174/gps.dat",
    callingham_dir / "gps.dat",
)
cache_url(
    "https://cdsarc.cds.unistra.fr/ftp/cats/J/ApJ/836/174/convex.dat",
    callingham_dir / "convex.dat",
)
cache_url(
    "https://cdsarc.cds.unistra.fr/ftp/cats/J/ApJ/836/174/pk72.dat",
    callingham_dir / "pk72.dat",
)

peaked_spectrum = pd.concat(
    [
        load_callingham_table(callingham_dir / "pkfreq.dat", sample="pkfreq", readme=callingham_readme),
        load_callingham_table(callingham_dir / "gps.dat", sample="gps", readme=callingham_readme),
        load_callingham_table(callingham_dir / "convex.dat", sample="convex", readme=callingham_readme),
        load_callingham_table(callingham_dir / "pk72.dat", sample="pk72", readme=callingham_readme),
    ],
    ignore_index=True,
)
peaked_spectrum = (
    peaked_spectrum.sort_values(
        ["Spk_Jy", "S200_Jy"], ascending=False, na_position="last", kind="mergesort"
    )
    .reset_index(drop=True)
)
peaked_spectrum = add_coord(peaked_spectrum)
peaked_spectrum = peaked_spectrum[
    [
        "name",
        "sample",
        "RA",
        "DEC",
        "coord",
        "Spk_Jy",
        "nuPk_MHz",
        "S200_Jy",
        "alpLow",
        "alpHigh",
        "alpTn",
        "alpTk",
        "q",
        "z",
    ]
]
print(
    f"Callingham+2017: {len(peaked_spectrum)} sources "
    f"(pkfreq+gps={int(peaked_spectrum['sample'].isin(['pkfreq', 'gps']).sum())})"
)
print(peaked_spectrum["sample"].value_counts().to_string())
show_table(peaked_spectrum)


cached /fast/claw/catalogs/callingham2017/ReadMe (34,631 bytes)
cached /fast/claw/catalogs/callingham2017/pkfreq.dat (983,710 bytes)
cached /fast/claw/catalogs/callingham2017/gps.dat (193,401 bytes)
cached /fast/claw/catalogs/callingham2017/convex.dat (85,956 bytes)
cached /fast/claw/catalogs/callingham2017/pk72.dat (26,676 bytes)
Callingham+2017: 1635 sources (pkfreq+gps=1483)
sample
pkfreq    1222
gps        261
convex     116
pk72        36
1635 rows, columns: ['name', 'sample', 'RA', 'DEC', 'coord', 'Spk_Jy', 'nuPk_MHz', 'S200_Jy', 'alpLow', 'alpHigh', 'alpTn', 'alpTk', 'q', 'z']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=              ...)

## Supernova Remnants

[Green's Catalogue of Galactic Supernova Remnants](https://www.mrao.cam.ac.uk/surveys/snrs/)
(2024 October version; 310 remnants). Machine-readable summary from VizieR
[`VII/297`](https://cdsarc.cds.unistra.fr/viz-bin/cat/VII/297)
(Green 2025, JApA, 46, 14).

Sorted by 1 GHz flux density (brightest first). The full table is `supernova_remnants`.
Cite Green (2025) and the 2024 October web catalogue if you use these positions.


In [7]:
snr_dir = CACHE_DIR / "green_snr"
cache_url("https://cdsarc.cds.unistra.fr/ftp/cats/VII/297/ReadMe", snr_dir / "ReadMe")
cache_url("https://cdsarc.cds.unistra.fr/ftp/cats/VII/297/snrs.dat", snr_dir / "snrs.dat")
snr = read_cds(snr_dir / "snrs.dat", snr_dir / "ReadMe")
sign = np.where(np.asarray(snr["DE-"], dtype=str) == "-", -1.0, 1.0)
ra = (np.asarray(snr["RAh"], dtype=float) + np.asarray(snr["RAm"], dtype=float) / 60.0
      + np.asarray(snr["RAs"], dtype=float) / 3600.0) * 15.0
dec = sign * (
    np.asarray(snr["DEd"], dtype=float) + np.asarray(snr["DEm"], dtype=float) / 60.0
)
other = pd.Series(np.asarray(snr["Names"], dtype=str)).str.strip()
supernova_remnants = pd.DataFrame(
    {
        "name": np.asarray(snr["SNR"], dtype=str),
        "other_names": other,
        "RA": ra,
        "DEC": dec,
        "type": pd.Series(np.asarray(snr["type"], dtype=str)).str.strip(),
        "MajDiam_arcmin": np.asarray(snr["MajDiam"], dtype=float),
        "MinDiam_arcmin": np.asarray(snr["MinDiam"], dtype=float),
        "S_1GHz_Jy": np.asarray(snr["S(1GHz)"], dtype=float),
        "sp_index": np.asarray(snr["Sp-Index"], dtype=float),
    }
)
supernova_remnants = (
    supernova_remnants.sort_values(
        "S_1GHz_Jy", ascending=False, na_position="last", kind="mergesort"
    )
    .reset_index(drop=True)
)
supernova_remnants = add_coord(supernova_remnants)
show_table(supernova_remnants)


cached /fast/claw/catalogs/green_snr/ReadMe (9,414 bytes)
cached /fast/claw/catalogs/green_snr/snrs.dat (19,799 bytes)
310 rows, columns: ['name', 'other_names', 'RA', 'DEC', 'type', 'MajDiam_arcmin', 'MinDiam_arcmin', 'S_1GHz_Jy', 'sp_index', 'coord']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=            name          ...)

## Pulsars

The [ATNF Pulsar Catalogue](https://www.atnf.csiro.au/research/pulsar/psrcat/)
(Manchester et al. 2005; live `psrcat.db` from the current public package).
Positions are ICRS: `RAJ`/`DECJ` when present, otherwise ecliptic
`ELONG`/`ELAT` converted to ICRS.

Sorted by YMW16 DM-distance (`DIST_DM`, kpc), nearest first. The full table is
`pulsars`. Cite Manchester et al. (2005) and the ATNF web catalogue.


In [8]:
def _psrcat_first_value(block: str) -> dict[str, str]:
    rec: dict[str, str] = {}
    for line in block.splitlines():
        if not line or line.startswith("#"):
            continue
        match = re.match(r"^([A-Z0-9_]+)\s+(\S+)", line)
        if match is None:
            continue
        rec.setdefault(match.group(1), match.group(2))
    return rec


def load_atnf_psrcat(db_path: Path) -> pd.DataFrame:
    text = db_path.read_text(encoding="latin1")
    rows: list[dict[str, object]] = []
    for block in re.split(r"\n@.*\n", text):
        rec = _psrcat_first_value(block)
        name = rec.get("PSRJ") or rec.get("PSRB")
        if not name:
            continue
        ra = dec = np.nan
        if "RAJ" in rec and "DECJ" in rec:
            sky = SkyCoord(rec["RAJ"], rec["DECJ"], unit=(u.hourangle, u.deg), frame="icrs")
            ra, dec = float(sky.ra.deg), float(sky.dec.deg)
        elif "ELONG" in rec and "ELAT" in rec:
            sky = SkyCoord(
                lon=float(rec["ELONG"]) * u.deg,
                lat=float(rec["ELAT"]) * u.deg,
                frame=BarycentricMeanEcliptic(),
            ).icrs
            ra, dec = float(sky.ra.deg), float(sky.dec.deg)
        else:
            continue
        p0 = rec.get("P0")
        if p0 is None and "F0" in rec:
            f0 = float(rec["F0"])
            p0 = f"{1.0 / f0:.12g}" if f0 else None
        dist = rec.get("DIST") or rec.get("DIST_DM")
        rows.append(
            {
                "name": name,
                "RA": ra,
                "DEC": dec,
                "P0_s": pd.to_numeric(p0, errors="coerce"),
                "DM": pd.to_numeric(rec.get("DM"), errors="coerce"),
                "S400_mJy": pd.to_numeric(rec.get("S400"), errors="coerce"),
                "S1400_mJy": pd.to_numeric(rec.get("S1400"), errors="coerce"),
                "Dist_kpc": pd.to_numeric(dist, errors="coerce"),
                "Type": rec.get("TYPE", ""),
                "Assoc": rec.get("ASSOC", ""),
            }
        )
    return pd.DataFrame(rows)


psrcat_dir = CACHE_DIR / "psrcat"
pkg = cache_url(
    "https://www.atnf.csiro.au/research/pulsar/psrcat/downloads/psrcat_pkg.tar.gz",
    psrcat_dir / "psrcat_pkg.tar.gz",
)
db_path = psrcat_dir / "psrcat.db"
if not db_path.is_file() or db_path.stat().st_size == 0:
    with tarfile.open(pkg, mode="r:gz") as tf:
        member = next(m for m in tf.getmembers() if m.name.endswith("psrcat.db"))
        extracted = tf.extractfile(member)
        if extracted is None:
            raise FileNotFoundError("psrcat.db missing from ATNF tarball")
        db_path.write_bytes(extracted.read())
    print(f"extracted {db_path} ({db_path.stat().st_size:,} bytes)")
else:
    print(f"cached {db_path} ({db_path.stat().st_size:,} bytes)")

header = db_path.read_text(encoding="latin1", errors="replace").splitlines()[0]
print("ATNF", header.lstrip("#").strip())

pulsars = load_atnf_psrcat(db_path)
pulsars = pulsars.sort_values("Dist_kpc", na_position="last", kind="mergesort").reset_index(drop=True)
pulsars = add_coord(pulsars)
show_table(pulsars)


cached /fast/claw/catalogs/psrcat/psrcat_pkg.tar.gz (1,224,627 bytes)
cached /fast/claw/catalogs/psrcat/psrcat.db (10,432,859 bytes)
ATNF CATALOGUE 2.8.1
4393 rows, columns: ['name', 'RA', 'DEC', 'P0_s', 'DM', 'S400_mJy', 'S1400_mJy', 'Dist_kpc', 'Type', 'Assoc', 'coord']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=            name          ...)

## Pulsar Wind Nebulae

The [Pulsar Wind Nebula Catalog](https://www.physics.mcgill.ca/~pulsar/pwncat.html)
(Roberts 2004; March 2005 version; extension of Kaspi, Roberts & Harding 2006)
lists PWNe with and without a detected pulsar. Positions are ICRS: ATNF
coordinates when the associated pulsar is in `pulsars`, otherwise the Galactic
`Gname` center (`Glll.l±bb.b` → Galactic → ICRS).

Sorted by `log_Edot` (highest first; PWNe without a measured Ė last). The full
table is `pulsar_wind_nebulae`. Cite Roberts (2004) / the McGill PWNCat page.


In [9]:
_TAG_RE = re.compile(r"(?is)<[^>]+>")
_WS_RE = re.compile(r"\s+")
_GNAME_RE = re.compile(r"G(?P<l>\d+(?:\.\d+)?)\s*(?P<b>[+-]\d+(?:\.\d+)?)")
_HTML_TABLE_RE = re.compile(r"(?is)<table[^>]*>(.*?)</table>")
_HTML_TR_RE = re.compile(r"(?is)<tr[^>]*>(.*?)</tr>")
_HTML_CELL_RE = re.compile(r"(?is)<t[dh][^>]*>(.*?)</t[dh]>")


def _strip_html(text: str) -> str:
    text = _TAG_RE.sub(" ", text)
    return _WS_RE.sub(" ", text).replace("\xa0", " ").strip()


def parse_html_tables(html: str) -> list[pd.DataFrame]:
    '''Parse HTML <table> elements with the stdlib (no lxml).'''
    tables: list[pd.DataFrame] = []
    for match in _HTML_TABLE_RE.finditer(html):
        rows: list[list[str]] = []
        for tr in _HTML_TR_RE.finditer(match.group(1)):
            cells = [_strip_html(c) for c in _HTML_CELL_RE.findall(tr.group(1))]
            if cells:
                rows.append(cells)
        if len(rows) < 2:
            continue
        header = rows[0]
        data = []
        for row in rows[1:]:
            if len(row) < len(header):
                row = row + [""] * (len(header) - len(row))
            else:
                row = row[: len(header)]
            data.append(row)
        tables.append(pd.DataFrame(data, columns=header))
    return tables


def gname_to_icrs(gname: str) -> tuple[float, float]:
    '''Convert a Galactic SNR/PWN Gname (Glll.l±bb.b) to ICRS degrees.'''
    match = _GNAME_RE.search(str(gname))
    if match is None:
        return float("nan"), float("nan")
    sky = SkyCoord(
        l=float(match.group("l")) * u.deg,
        b=float(match.group("b")) * u.deg,
        frame="galactic",
    ).icrs
    return float(sky.ra.deg), float(sky.dec.deg)


def _display_name(other: str, pulsar: str, gname: str) -> str:
    other = str(other).strip()
    if other:
        return other.split(",")[0].strip()
    pulsar = str(pulsar).strip()
    if pulsar:
        return pulsar
    return str(gname).strip()


pwn_html = cache_url(
    "https://www.physics.mcgill.ca/~pulsar/pwncat.html",
    CACHE_DIR / "pwncat" / "pwncat.html",
)
pwn_tables = parse_html_tables(pwn_html.read_text(encoding="latin1", errors="replace"))
if len(pwn_tables) < 2:
    raise RuntimeError(f"expected 2 PWNCat tables, got {len(pwn_tables)}")
pwn_with_psr, pwn_no_psr = pwn_tables[0], pwn_tables[1]

atnf_by_name = pulsars.set_index("name", drop=False)


def _pwn_rows(frame: pd.DataFrame, *, has_pulsar: bool) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for rec in frame.to_dict(orient="records"):
        pulsar = str(rec.get("Pulsar", "") or "").strip()
        gname = str(rec.get("Gname", "") or "").strip()
        other = str(rec.get("other name(s)", "") or "").strip()
        ra_g, dec_g = gname_to_icrs(gname)
        ra, dec, pos_source = ra_g, dec_g, "Gname"
        if has_pulsar and pulsar and pulsar in atnf_by_name.index:
            hit = atnf_by_name.loc[pulsar]
            if isinstance(hit, pd.DataFrame):
                hit = hit.iloc[0]
            ra, dec, pos_source = float(hit["RA"]), float(hit["DEC"]), "ATNF"
        log_edot = pd.to_numeric(rec.get("log Edot"), errors="coerce") if has_pulsar else np.nan
        rows.append(
            {
                "name": _display_name(other, pulsar, gname),
                "pulsar": pulsar,
                "Gname": gname,
                "other_names": other,
                "has_pulsar": has_pulsar,
                "log_Edot": float(log_edot) if pd.notna(log_edot) else np.nan,
                "RA": ra,
                "DEC": dec,
                "pos_source": pos_source,
                "rP": str(rec.get("rP", "") or "").strip(),
                "xP": str(rec.get("xP", "") or "").strip(),
                "gP": str(rec.get("gP", "") or "").strip(),
                "R": str(rec.get("R", "") or "").strip(),
                "X": str(rec.get("X", "") or "").strip(),
                "O": str(rec.get("O", "") or "").strip(),
                "G": str(rec.get("G", "") or "").strip(),
            }
        )
    return rows


pulsar_wind_nebulae = pd.DataFrame(
    _pwn_rows(pwn_with_psr, has_pulsar=True) + _pwn_rows(pwn_no_psr, has_pulsar=False)
)
pulsar_wind_nebulae = (
    pulsar_wind_nebulae.sort_values("log_Edot", ascending=False, na_position="last", kind="mergesort")
    .reset_index(drop=True)
)
pulsar_wind_nebulae = add_coord(pulsar_wind_nebulae)
pulsar_wind_nebulae = pulsar_wind_nebulae[
    [
        "name",
        "pulsar",
        "Gname",
        "other_names",
        "has_pulsar",
        "log_Edot",
        "RA",
        "DEC",
        "coord",
        "pos_source",
        "rP",
        "xP",
        "gP",
        "R",
        "X",
        "O",
        "G",
    ]
]
print(
    f"PWNCat: {len(pulsar_wind_nebulae)} nebulae "
    f"({int(pulsar_wind_nebulae['has_pulsar'].sum())} with pulsar, "
    f"{int((~pulsar_wind_nebulae['has_pulsar']).sum())} without)"
)
print("pos_source counts:")
print(pulsar_wind_nebulae["pos_source"].value_counts().to_string())
show_table(pulsar_wind_nebulae)


cached /fast/claw/catalogs/pwncat/pwncat.html (75,002 bytes)
PWNCat: 56 nebulae (32 with pulsar, 24 without)
pos_source counts:
pos_source
ATNF     32
Gname    24
56 rows, columns: ['name', 'pulsar', 'Gname', 'other_names', 'has_pulsar', 'log_Edot', 'RA', 'DEC', 'coord', 'pos_source', 'rP', 'xP', 'gP', 'R', 'X', 'O', 'G']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=               name      p...)

## X-ray binaries

Galactic low- and high-mass X-ray binaries from the Tübingen
[X-ray Binary Catalogues](http://astro.uni-tuebingen.de/~xrbcat/)
([LMXBcat](http://astro.uni-tuebingen.de/~xrbcat/LMXBcat/january_24/LMXBcat.csv),
January 2024; [HMXBcat](http://astro.uni-tuebingen.de/~xrbcat/HMXBcat/March2025/HMXBcat.csv),
March 2025). Semicolon-separated CSVs with ICRS `RAdeg`/`DEdeg`.

Combined table is `xray_binaries` (`class` = `LMXB` or `HMXB`), sorted by
`Dist_pc` (nearest first). Paste `coord` into `metacatalog_query.ipynb`.


In [10]:
def load_tuebingen_xrb(path: Path, *, xrb_class: str) -> pd.DataFrame:
    '''Read a Tübingen LMXB/HMXB CSV (semicolon-separated) into a common schema.'''
    raw = pd.read_csv(path, sep=";")

    def col(name: str) -> pd.Series:
        if name in raw.columns:
            return raw[name]
        return pd.Series(pd.NA, index=raw.index)

    out = pd.DataFrame(
        {
            "name": raw["Name"].astype(str).str.strip(),
            "class": xrb_class,
            "Alt_Name": col("Alt_Name").fillna("").astype(str).str.strip(),
            "RA": pd.to_numeric(raw["RAdeg"], errors="coerce"),
            "DEC": pd.to_numeric(raw["DEdeg"], errors="coerce"),
            "PosErr_arcsec": pd.to_numeric(raw["PosErr"], errors="coerce"),
            "Xray_Type": col("Xray_Type").fillna("").astype(str).str.strip(),
            "SpType": col("SpType").fillna("").astype(str).str.strip(),
            "Porb_d": pd.to_numeric(col("Porb"), errors="coerce"),
            "Ppulse_s": pd.to_numeric(col("Ppulse"), errors="coerce"),
            "Dist_pc": pd.to_numeric(col("Mean_Dist"), errors="coerce"),
            "Gmag": pd.to_numeric(col("Gmag"), errors="coerce"),
            "Vmag": pd.to_numeric(col("Vmag"), errors="coerce"),
        }
    )
    return out.dropna(subset=["RA", "DEC"]).reset_index(drop=True)


xrb_dir = CACHE_DIR / "xrbcat"
lmxb_path = cache_url(
    "http://astro.uni-tuebingen.de/~xrbcat/LMXBcat/january_24/LMXBcat.csv",
    xrb_dir / "LMXBcat.csv",
)
hmxb_path = cache_url(
    "http://astro.uni-tuebingen.de/~xrbcat/HMXBcat/March2025/HMXBcat.csv",
    xrb_dir / "HMXBcat.csv",
)
xray_binaries = pd.concat(
    [
        load_tuebingen_xrb(lmxb_path, xrb_class="LMXB"),
        load_tuebingen_xrb(hmxb_path, xrb_class="HMXB"),
    ],
    ignore_index=True,
)
xray_binaries = (
    xray_binaries.sort_values("Dist_pc", na_position="last", kind="mergesort")
    .reset_index(drop=True)
)
xray_binaries = add_coord(xray_binaries)
xray_binaries = xray_binaries[
    [
        "name",
        "class",
        "Alt_Name",
        "RA",
        "DEC",
        "coord",
        "PosErr_arcsec",
        "Xray_Type",
        "SpType",
        "Porb_d",
        "Ppulse_s",
        "Dist_pc",
        "Gmag",
        "Vmag",
    ]
]
print(
    f"X-ray binaries: {len(xray_binaries)} "
    f"(LMXB={int((xray_binaries['class'] == 'LMXB').sum())}, "
    f"HMXB={int((xray_binaries['class'] == 'HMXB').sum())})"
)
print("class counts:")
print(xray_binaries["class"].value_counts().to_string())
show_table(xray_binaries)


cached /fast/claw/catalogs/xrbcat/LMXBcat.csv (167,859 bytes)
cached /fast/claw/catalogs/xrbcat/HMXBcat.csv (144,314 bytes)
X-ray binaries: 533 (LMXB=360, HMXB=173)
class counts:
class
LMXB    360
HMXB    173
533 rows, columns: ['name', 'class', 'Alt_Name', 'RA', 'DEC', 'coord', 'PosErr_arcsec', 'Xray_Type', 'SpType', 'Porb_d', 'Ppulse_s', 'Dist_pc', 'Gmag', 'Vmag']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=                name class...)

## X-ray/optical systems

Galactic compact-object and related binaries from Table 2 of
[Rodriguez 2024, arXiv:2401.09537](https://arxiv.org/html/2401.09537v1#A2)
(*From Active Stars to Black Holes*). These are the literature systems plotted
on the X-ray Main Sequence (redbacks, black widows, LMXBs, HMXBs, symbiotic
stars, supersoft sources).

Gaia DR3 ICRS positions are included with each row (resolved once from ESA TAP).
The table is `xray_optical`, in the paper's class order.


In [11]:
# Rodriguez 2024 Table 2, with Gaia DR3 ICRS positions (queried once from ESA TAP).
# xray_ref: (1) spider binaries; (2) BH LMXBs; (3) NS LMXBs; (4) HMXBs;
# (5) symbiotic NS; (6) symbiotic WD; (7) 4XMM-DR13 SSS.
XRAY_OPTICAL_ROWS = [
    ("J0212+5320", "Redback", 455282205716288384, 33.043636, 53.360781, 1),
    ("J1048+2339", "Redback", 3990037124929068032, 162.180900, 23.664831, 1),
    ("J1306-40", "Redback", 6140785016794586752, 196.734467, -40.589833, 1),
    ("J1431-4715", "Redback", 6098156298150016768, 217.935887, -47.257676, 1),
    ("J1622-0315", "Redback", 4358428942492430336, 245.748445, -3.260352, 1),
    ("J1628-3205", "Redback", 6025344817107454464, 247.029176, -32.096950, 1),
    ("J1723-2837", "Redback", 4059795674516044800, 260.846581, -28.632665, 1),
    ("J1803-6707", "Redback", 6436867623955512064, 270.767647, -67.126710, 1),
    ("J1816+4510", "Redback", 2115337192179377792, 274.149726, 45.176069, 1),
    ("J1908+2105", "Redback", 4519819661567533696, 287.238717, 21.083920, 1),
    ("J1910-5320", "Redback", 6644467032871428992, 287.704669, -53.349200, 1),
    ("J2039-5618", "Redback", 6469722508861870080, 309.895702, -56.285910, 1),
    ("J2129-0429", "Redback", 2672030065446134656, 322.437749, -4.485225, 1),
    ("J2215+5135", "Redback", 2001168543319218048, 333.886196, 51.593455, 1),
    ("J2339-0533", "Redback", 2440660623886405504, 354.911440, -5.551465, 1),
    ("J1311-3430", "Black Widow", 6179115508262195200, 197.940506, -34.508438, 1),
    ("J1653-0158", "Black Widow", 4379227476242700928, 253.408555, -1.976915, 1),
    ("J1810+1744", "Black Widow", 4526229058440076288, 272.655365, 17.743711, 1),
    ("B1957+20", "Black Widow", 1823773960079216896, 299.903093, 20.804020, 1),
    ("GROJ0422+32", "LMXB (BH)", 172650748928103552, 65.428011, 32.907483, 2),
    ("A0620-00", "LMXB (BH)", 3118721026600835328, 95.685591, -0.345659, 2),
    ("V404 Cyg", "LMXB (BH)", 2056188624872569088, 306.015909, 33.867177, 2),
    ("XTE J1118+480", "LMXB (BH)", 789430249033567744, 169.544851, 48.036724, 2),
    ("GROJ1655-40", "LMXB (BH)", 5969790961312131456, 253.500568, -39.845800, 2),
    ("4U 2129+47", "LMXB (NS)", 1978241050130301312, 322.859207, 47.290123, 3),
    ("Cen X-4", "LMXB (NS)", 6205715168442046592, 224.591399, -31.669002, 3),
    ("Aql X-1", "LMXB (NS)", 4264296556603631872, 287.816897, 0.584941, 3),
    ("SAX J1808.4-3658", "LMXB (NS)", 4037867740522984832, 272.114284, -36.977908, 3),
    ("A0535+26", "HMXB (NS)", 3441207615229815040, 84.727392, 26.315775, 4),
    ("KS 1947+300", "HMXB (NS)", 2031939548802102656, 297.397839, 30.208808, 4),
    ("V4641 Sgr", "HMXB (BH)", 4053096388919082368, 274.840139, -25.407179, 4),
    ("Cyg X-1", "HMXB (BH)", 2059383668236814720, 299.590295, 35.201579, 4),
    ("GX 1+4", "Symbiotic (NS)", 4110236324513030656, 263.008955, -24.745600, 5),
    ("4U 1954+319", "Symbiotic (NS)", 2034031438383765760, 298.926398, 32.096930, 5),
    ("CXOGBS J173620.2-293338", "Symbiotic (NS)", 4060066227422719872, 264.084127, -29.560827, 5),
    ("4U 1700+24", "Symbiotic (NS)", 4571810378118789760, 256.643761, 23.971820, 5),
    ("NQ Gem", "Symbiotic (WD)", 868424696282795392, 112.977122, 24.503471, 6),
    ("UV Aur", "Symbiotic (WD)", 180919213811383680, 80.453799, 32.511146, 6),
    ("ZZ CMi", "Symbiotic (WD)", 3155368612444708096, 111.058322, 8.897700, 6),
    ("ER Del", "Symbiotic (WD)", 1750795043999682304, 310.693762, 8.687115, 6),
    ("CD -283719", "Symbiotic (WD)", 5608089951177429120, 105.288142, -29.106940, 6),
    ("RX J0019.8+2156", "SSS", 2800287654443977344, 4.958110, 21.947799, 7),
    ("RX J0925.7-4758", "SSS", 5422337322910734080, 141.441648, -47.971474, 7),
    ("RR Tel", "SSS", 6448785024330499456, 301.077269, -55.725891, 7),
]
xray_optical = pd.DataFrame(
    XRAY_OPTICAL_ROWS,
    columns=["name", "class", "gaia_source_id", "RA", "DEC", "xray_ref"],
)
xray_optical = add_coord(xray_optical)
xray_optical = xray_optical[
    ["name", "class", "gaia_source_id", "RA", "DEC", "coord", "xray_ref"]
]
print(f"{len(xray_optical)} systems; class counts:")
print(xray_optical["class"].value_counts().to_string())
show_table(xray_optical)


44 systems; class counts:
class
Redback           15
LMXB (BH)          5
Symbiotic (WD)     5
LMXB (NS)          4
Black Widow        4
Symbiotic (NS)     4
SSS                3
HMXB (NS)          2
HMXB (BH)          2
44 rows, columns: ['name', 'class', 'gaia_source_id', 'RA', 'DEC', 'coord', 'xray_ref']


Tabulator(disabled=True, header_filters=True, height=420, page_size=25, pagination='local', show_index=False, sizing_mode='stretch_width', value=              ...)

## Sample sizes

Kernel variables for later cells or for copying into `metacatalog_query.ipynb`:
`local_galaxies`, `hrs_gleam_galaxies`, `galaxy_clusters`, `giant_radio_sources`,
`peaked_spectrum`, `supernova_remnants`, `pulsars`, `pulsar_wind_nebulae`,
`xray_binaries`, `xray_optical`.


In [12]:
summary = pd.DataFrame(
    [
        {"sample": "local_galaxies", "n": len(local_galaxies), "sort": "DistMpc", "note": f"NED-LVS G, z<={MAX_REDSHIFT_GALAXIES}, D>={MIN_DIST_MPC} Mpc, Mstar|SFR"},
        {"sample": "hrs_gleam_galaxies", "n": len(hrs_gleam_galaxies), "sort": "Dist_Mpc", "note": "Takeuchi+2026 Table 1 (HRS×GLEAM)"},
        {"sample": "galaxy_clusters", "n": len(galaxy_clusters), "sort": "z", "note": "MCXC"},
        {"sample": "giant_radio_sources", "n": len(giant_radio_sources), "sort": "D_Mpc", "note": "Kuźmicz+2018 GRS" + ("" if MAX_REDSHIFT_GRS is None else f", z<={MAX_REDSHIFT_GRS}")},
        {"sample": "peaked_spectrum", "n": len(peaked_spectrum), "sort": "Spk_Jy", "note": "Callingham+2017 GLEAM"},
        {"sample": "supernova_remnants", "n": len(supernova_remnants), "sort": "S_1GHz_Jy", "note": "Green 2024 Oct"},
        {"sample": "pulsars", "n": len(pulsars), "sort": "Dist_kpc", "note": "ATNF psrcat"},
        {"sample": "pulsar_wind_nebulae", "n": len(pulsar_wind_nebulae), "sort": "log_Edot", "note": "Roberts PWNCat"},
        {"sample": "xray_binaries", "n": len(xray_binaries), "sort": "Dist_pc", "note": "Tübingen LMXB+HMXB"},
        {"sample": "xray_optical", "n": len(xray_optical), "sort": "paper order", "note": "Rodriguez 2024 Table 2"},
    ]
)
show_table(summary)


10 rows, columns: ['sample', 'n', 'sort', 'note']


Tabulator(disabled=True, header_filters=True, height=360, page_size=10, show_index=False, sizing_mode='stretch_width', value=                sample    ...)